Duomenų įkėlimas iš Kaggle svetainės.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("samuelcortinhas/muffin-vs-chihuahua-image-classification")

print("Path to dataset files:", path)

import os

print(os.listdir(path))

Sukuriam naujus aplankus ir apjungiam test su train duomenimis.

In [ ]:
import shutil
import os

train_dir = os.path.join(path, 'train')
test_dir = os.path.join(path, 'test')
combined_dir = '/kaggle/working/all_data'

os.makedirs(combined_dir, exist_ok=True)

# Sukuriam klasių aplankus
for class_name in os.listdir(train_dir):
    os.makedirs(os.path.join(combined_dir, class_name), exist_ok=True)

# Perkopijuojam visus train
for class_name in os.listdir(train_dir):
    src = os.path.join(train_dir, class_name)
    dst = os.path.join(combined_dir, class_name)
    for file in os.listdir(src):
        shutil.copy(os.path.join(src, file), os.path.join(dst, file))

# Perkopijuojam visus test
for class_name in os.listdir(test_dir):
    src = os.path.join(test_dir, class_name)
    dst = os.path.join(combined_dir, class_name)
    for file in os.listdir(src):
        shutil.copy(os.path.join(src, file), os.path.join(dst, file))

print("Visi failai sujungti į:", combined_dir)

Daliname duomenis į mokymo, validavimo ir testavimo aibes (80:10:10).

3 aplankai su 2 aplankais viduje (chihuahua ir muffin).

data_split/

 ├── train/
 │   ├── muffin/
 │   ├── chihuahua/

 ├── val/
 │   ├── muffin/
 │   ├── chihuahua/

 └── test/
     ├── muffin/
     ├── chihuahua/

In [ ]:
!pip install split-folders

import splitfolders

# Dalinam 80:10:10
splitfolders.ratio(
    combined_dir,
    output="/kaggle/working/data_split",
    seed=42,
    ratio=(.8, .1, .1)
)


base_dir = "/kaggle/working/data_split"
total_counts = {}

for split in ['train', 'val', 'test']:
    split_path = os.path.join(base_dir, split)
    print(f"\n {split.upper()}:")

    # kiekvienai klasei
    split_total = 0
    for class_name in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, class_name)
        if os.path.isdir(class_path):
            num_images = len([
                f for f in os.listdir(class_path)
                if os.path.isfile(os.path.join(class_path, f))
            ])
            print(f"  {class_name}: {num_images} paveikslėlių")
            split_total += num_images

    total_counts[split] = split_total
    print(f"Iš viso {split}: {split_total} paveikslėlių")



Įkeliame reikalingas bibliotekas.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import matplotlib.pyplot as plt

**Duomenų paruošimas**

Sukuriama duomenų generavimo funkcija.

Suskirstoma automatiškai į klases, dėl ImageDataGenerator - chihuahua klasė 0, muffin klasė 1.

* target_size=(150,150) → kiekviena nuotrauka paverčiama į formą (150,150,3)

* rescale=1./255 → pikselių reikšmės normalizuojamos į [0,1] intervalą

* Iš to gaunami du numpy masyvai: (X_batch, y_batch)

Modelis gauna nuotraukas, paverstas į normalizuotus RGB pikselių masyvus.




In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def create_generators(class_mode, batch_size, img_size=(150,150)):
    datagen = ImageDataGenerator(rescale=1./255)

    train_gen = datagen.flow_from_directory(
        '/kaggle/working/data_split/train/',
        target_size=img_size,
        batch_size=batch_size,
        class_mode=class_mode
    )
    val_gen = datagen.flow_from_directory(
        '/kaggle/working/data_split/val/',
        target_size=img_size,
        batch_size=batch_size,
        class_mode=class_mode
    )
    test_gen = datagen.flow_from_directory(
        '/kaggle/working/data_split/test/',
        target_size=img_size,
        batch_size=batch_size,
        class_mode=class_mode,
        shuffle=False
    )


    return train_gen, val_gen, test_gen


Generuojame duomenų aibes.

In [ ]:
train_gen, val_gen, test_gen = create_generators(class_mode='binary', batch_size=32)

**Modelio mokymo funkcija**

In [ ]:
def train_model(model, train_gen, val_gen, epochs):
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs
    )
    return history


**Skirtingos architektūros:**


  Modelis simple

In [ ]:
model1 = models.Sequential()
model1.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
model1.add(layers.Flatten())
model1.add(layers.Dense(1, activation='sigmoid'))

model1.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history1 = train_model(model1, train_gen, val_gen, epochs=10)


Modelis medium

In [ ]:
model2 = models.Sequential()
model2.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
model2.add(layers.MaxPooling2D((2, 2)))
model2.add(layers.Conv2D(64, (3, 3), activation='relu'))
model2.add(layers.MaxPooling2D((2, 2)))
model2.add(layers.Flatten())
model2.add(layers.Dense(128, activation='relu'))
model2.add(layers.Dense(1, activation='sigmoid'))


model2.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history2 = train_model(model2, train_gen, val_gen, epochs=10)

Modelis deep

In [ ]:
model3 = models.Sequential()
model3.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
model3.add(layers.MaxPooling2D((2, 2)))
model3.add(layers.Conv2D(64, (3, 3), activation='relu'))
model3.add(layers.MaxPooling2D((2, 2)))
model3.add(layers.Conv2D(128, (3, 3), activation='relu'))
model3.add(layers.Conv2D(256, (3, 3), activation='relu'))
model3.add(layers.MaxPooling2D((2, 2)))
model3.add(layers.Flatten())
model3.add(layers.Dense(512, activation='relu'))
model3.add(layers.Dense(1, activation='sigmoid'))

model3.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history3 = train_model(model3, train_gen, val_gen, epochs=10)

Treniruojam visus 3 modelius

Brėžiam jiems grafikus

In [ ]:
history = [history1, history2, history3]
for i in range(3):
    plt.figure()
    plt.plot(history[i].history['accuracy'], label='Mokymo')
    plt.plot(history[i].history['val_accuracy'], label='Validavimo')
    plt.xlabel('Epocha')
    plt.ylabel('Tikslumas')
    plt.ylim([0.5, 1])
    plt.xlim([-0.5, 9.5])
    plt.title(f"Modelis {i+1}")
    plt.legend(loc='lower right')
    plt.show()

In [ ]:
for i in range(3):
    plt.figure()
    plt.plot(history[i].history['loss'], label='Mokymo')
    plt.plot(history[i].history['val_loss'], label='Validavimo')
    plt.xlabel('Epocha')
    plt.ylabel('Paklaida')
    plt.ylim([0, 1.2])
    plt.xlim([-0.5, 9.5])
    plt.title(f"Modelis {i+1}")
    plt.legend(loc='lower left')
    plt.show()

*Funkcija hiperparametrų keitimui*

Funkcija sukurta pagal trečiąją modelio architektūrą.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, SGD, Adagrad, RMSprop

def create_model3(
    conv_filters,
    activation,
    kernel_sizes,
    pool_sizes,
    dense_neurons,
    dropout_rates,  
    optimizer_type,
    learning_rate,
    loss_type,
    use_batchnorm,
    input_shape=(150,150,3)
):
    model = models.Sequential()

    # Pirmas konvoliucijos sluoksnis
    model.add(layers.Conv2D(conv_filters[0], kernel_sizes[0], padding='same', input_shape=input_shape))
    if use_batchnorm:
        model.add(layers.BatchNormalization())
    model.add(layers.Activation(activation))
    model.add(layers.MaxPooling2D(pool_size=pool_sizes[0]))
    if dropout_rates[0] > 0:
        model.add(layers.Dropout(dropout_rates[0]))

    # Likę konvoliucijos sluoksniai
    for i in range(1, len(conv_filters)):
        model.add(layers.Conv2D(conv_filters[i], kernel_sizes[i], padding='same'))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation(activation))
        if i-1 < len(pool_sizes):
            model.add(layers.MaxPooling2D(pool_size=pool_sizes[i-1]))
        if dropout_rates[i] > 0:
            model.add(layers.Dropout(dropout_rates[i]))

    model.add(layers.Flatten())

    # pilnai sujungti sluoksniai
    for j, neurons in enumerate(dense_neurons):
        model.add(layers.Dense(neurons))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation(activation))
        if dropout_rates[len(conv_filters)+j] > 0:
            model.add(layers.Dropout(dropout_rates[len(conv_filters)+j]))

    # Išėjimo sluoksnis
    model.add(layers.Dense(1, activation='sigmoid'))

    optimizer_type = optimizer_type.lower()
    if optimizer_type == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer_type == 'sgd':
        opt = SGD(learning_rate=learning_rate)
    elif optimizer_type == 'adagrad':
        opt = Adagrad(learning_rate=learning_rate)
    elif optimizer_type == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=opt, loss=loss_type, metrics=['accuracy'])

    return model


**Funkcija grafikų braižymui**

In [ ]:
def grafikai(history):
  plt.figure()
  plt.plot(history.history['loss'], label='Mokymo')
  plt.plot(history.history['val_loss'], label='Validavimo')
  plt.xlabel('Epocha')
  plt.ylabel('Paklaida')
  plt.xlim([-0.5, 9.5])
  plt.legend(loc='lower left')
  plt.show()

  plt.figure()
  plt.plot(history.history['accuracy'], label='Mokymo')
  plt.plot(history.history['val_accuracy'], label='Validavimo')
  plt.xlabel('Epocha')
  plt.ylabel('Tikslumas')
  plt.ylim([0, 1])
  plt.xlim([-0.5, 9.5])
  plt.legend(loc='lower right')
  plt.show()

Pridedame vieną išmetimo (dropout) sluoksnį po pirmo konvoliucijos sluoksnio ir vieną  po pilnai sujungto (dense) sluoksnio.

Keičiame jų tikimybes: 0.3, 0.6, 0.9.

Brėžiame grafikus kiekvienam tinklo modeliui.

In [ ]:

m1 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.3, 0, 0, 0, 0.3],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h1 = train_model(m1, train_gen, val_gen, epochs=10)
grafikai(h1)

In [ ]:
m2 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h2 = train_model(m2, train_gen, val_gen, epochs=10)
grafikai(h2)

In [ ]:
m3 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.9, 0, 0, 0, 0.9],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h3 = train_model(m3, train_gen, val_gen, epochs=10)
grafikai(h3)

Pridedame paketų normalizavimą.

(Be normalizavimo, tai tiesiog m2 modelis).

In [ ]:
m4 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=True,
    input_shape=(150,150,3)
)
h4 = train_model(m4, train_gen, val_gen, epochs=10)
grafikai(h4)

Keičiam aktyvacijos funkcijas.

In [ ]:
m5 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='tanh',
    kernel_sizes=[(3,3), (3,3), (3, 3,), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h5 = train_model(m5, train_gen, val_gen, epochs=10)
grafikai(h5)

In [ ]:

plt.figure()
plt.plot(h5.history['accuracy'], label='Mokymo')
plt.plot(h5.history['val_accuracy'], label='Validavimo')
plt.xlabel('Epocha')
plt.ylabel('Tikslumas')
plt.ylim([0, 1])
plt.xlim([-0.5, 9.5])
plt.legend(loc='lower right')
plt.show()

LeakyReLu

In [ ]:
m6 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='leaky_relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h6 = train_model(m6, train_gen, val_gen, epochs=10)
grafikai(h6)

Softplus

In [ ]:
m11 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='softplus',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h11= train_model(m11, train_gen, val_gen, epochs=10)
grafikai(h11)

Sigmoidinė

In [ ]:
m7 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='sigmoid',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adam',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h7 = train_model(m7, train_gen, val_gen, epochs=10)
grafikai(h7)

In [ ]:
plt.figure()
plt.plot(h8.history['accuracy'], label='Mokymo')
plt.plot(h8.history['val_accuracy'], label='Validavimo')
plt.xlabel('Epocha')
plt.ylabel('Tikslumas')
plt.ylim([0, 1])
plt.xlim([-0.5, 9.5])
plt.legend(loc='lower right')
plt.show()

Keičiame optimizavimo metodą: Adam, SGD, Adagrad, RMSprop.


In [ ]:
m8 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='sgd',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h8 = train_model(m8, train_gen, val_gen, epochs=10)
grafikai(h8)

In [ ]:
m9 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='adagrad',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h9 = train_model(m9, train_gen, val_gen, epochs=10)
grafikai(h9)

In [ ]:
grafikai(h2)

In [ ]:
m10 = create_model3(
    conv_filters=[32, 64, 128, 256],
    activation='relu',
    kernel_sizes=[(3,3), (3,3), (3, 3), (3, 3)],
    pool_sizes=[(2,2), (2,2), (2,2)],
    dense_neurons=[512],
    dropout_rates=[0.6, 0, 0, 0, 0.6],
    optimizer_type='rmsprop',
    learning_rate=0.001,
    loss_type='binary_crossentropy',
    use_batchnorm=False,
    input_shape=(150,150,3)
)
h10 = train_model(m10, train_gen, val_gen, epochs=10)
grafikai(h10)

Testuojamas geriausias variantas (m2).

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

def evaluate_model(model, test_gen):
    pred = model.predict(test_gen)

    if pred.shape[1] > 1:
        # multi-class
        y_pred = np.argmax(pred, axis=1)
    else:
        # binary
        y_pred = np.where(pred>0.5, 1, 0).flatten()

    y_true = test_gen.classes
    cm = confusion_matrix(y_true, y_pred)
    return cm, y_true, y_pred


In [ ]:
cm, y_true, y_pred = evaluate_model(m2, test_gen)
print(cm)

Klasifikavio matrica

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

class_names = ['čichuachua', 'keksiukas']
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Tikros klasės')
plt.xlabel('Modelio gautos klasės')
plt.title('Klasifikavimo matrica')
plt.show()


Išsitraukiam 30 reikšmių, jas palyginame su modelio išėjimo reikšmėmis.

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


filenames = np.array(test_gen.filenames)
true_labels = np.array(test_gen.classes)

class_indices = test_gen.class_indices
print("Klasės:", class_indices)

sample_indices = []
for cls_name, cls_idx in class_indices.items():
    cls_idxs = np.where(true_labels == cls_idx)[0]      # visi tos klasės paveikslėliai
    np.random.shuffle(cls_idxs)
    sample_indices.extend(cls_idxs[:15])                # pvz., 15 iš kiekvienos klasės

sample_indices = np.array(sample_indices[:30])

# 4. Sudedame į lentelę
results = pd.DataFrame({
    'tikra_klase': true_labels[sample_indices],
    'prognoze': y_pred[sample_indices]
})

print(results.head(30))


Klasifikavimo tikslumas ir paklaida testavimo duomenims.

In [ ]:

test_ev = m2.evaluate(test_gen, verbose=2)
print(test_ev)